In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

import subprocess
from scipy.io import mmread, mmwrite
from scipy.sparse import csr_matrix

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
barcodes_path = f'{projdir}/mtx/atac/chromvar/chromvar.cells.barcodes.tsv'
barcodes = pd.read_csv(barcodes_path, sep='\t', header=None)
barcodes.columns = ['tmp_bc']
barcodes.shape

In [ ]:
features_path = f'{projdir}/mtx/atac/chromvar/chromvar.cells.features.tsv'
features = pd.read_csv(features_path, sep='\t', header=None)
features.columns = ['TF']
features.shape

In [ ]:
mtx_path = f'{projdir}/mtx/atac/chromvar/chromvar.cells.mtx'
mtx = csr_matrix(mmread(mtx_path))

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/csv/clusters/rna_leiden_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
adata = anndata.AnnData(X=mtx, obs=barcodes, var=features)

In [ ]:
adata

In [ ]:
adata.obs.index = adata.obs['tmp_bc']
adata.obs['sample'] = [x.split('#')[0] for x in adata.obs.index]
adata.obs['barcode'] = [x.split('#')[1] for x in adata.obs.index]
adata.obs['final_barcode'] = [f'{x}_{y}' for x,y in zip(adata.obs['barcode'], adata.obs['sample'])]
adata.obs.index = adata.obs['final_barcode']

In [ ]:
adata.obs['coarse'] = clusters.reindex(adata.obs.index)['coarse_v7']
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['clusters_v7']
adata.obs['assignment'] = clusters.reindex(adata.obs.index)['assignment']
adata.obs['time_num'] = clusters.reindex(adata.obs.index)['time_num']

In [ ]:
adata.obs.head()

### write

In [ ]:
adata.write_h5ad(f'{projdir}/h5ad/atac/processed/chromvar/intersect_rna.h5ad')

In [ ]:
!date